# 3D U-Net Training Pipeline for Cell Tracking
This notebook trains a 3D U-Net (via MONAI) to predict Gaussian heatmaps of cell centers.
**Usage:** Run this on Kaggle with GPU T4x2 or P100 enabled. Save the resulting `model_best.pth` as a Kaggle Dataset to use in the inference notebook.



In [ ]:
!pip install -q "monai[ignite, torchvision]" blosc2


In [ ]:
import os
import json
import time
import math
import numpy as np
import pandas as pd
import blosc2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import monai
from monai.networks.nets import UNet
from monai.losses import FocalLoss
from monai.inferers import sliding_window_inference

print("MONAI version:", monai.__version__)
print("PyTorch version:", torch.__version__)



In [ ]:
# Configuration
TRAIN_DIR = '/kaggle/input/competitions/biohub-cell-tracking-during-development/train'
EPOCHS = 10
BATCH_SIZE = 1
LEARNING_RATE = 1e-4

# Patch size for training (Z, Y, X)
# The Z-axis is anisotropic (thinner), so we take a smaller patch in Z
PATCH_SIZE = (32, 128, 128)

# Gaussian heatmap generation parameters
HEATMAP_SIGMA = 2.0  # size of the Gaussian ball around each cell



In [ ]:
# Dataset Definition
class CellTrackingDataset(Dataset):
    def __init__(self, data_dir, patch_size, heatmap_sigma, is_train=True):
        self.data_dir = data_dir
        self.patch_size = patch_size
        self.heatmap_sigma = heatmap_sigma
        
        # Discover all samples
        self.samples = []
        for d in sorted(os.listdir(data_dir)):
            if d.endswith('.zarr'):
                ds_name = d.replace('.zarr', '')
                self.samples.append(ds_name)
        
        # Load ground truth coordinates
        gt_path = os.path.join(data_dir, '..', 'train.csv')
        if os.path.exists(gt_path):
            self.df = pd.read_csv(gt_path)
            self.df = self.df[self.df['row_type'] == 'node']
        else:
            self.df = None
            
        # We will train on random frames from the available datasets
        self.valid_frames = []
        if self.df is not None:
            for ds in self.samples:
                ds_data = self.df[self.df['dataset'] == ds]
                for t in ds_data['t'].unique():
                    self.valid_frames.append((ds, t))
                    
        print(f"Discovered {len(self.valid_frames)} valid frames for training.")

    def __len__(self):
        return len(self.valid_frames)

    def _generate_heatmap(self, coords, shape):
        """Generates a 3D volume with Gaussian blobs at given coordinates."""
        heatmap = np.zeros(shape, dtype=np.float32)
        Z, Y, X = shape
        
        # Limit the bounding box of the gaussian to speed up computation
        radius = int(math.ceil(3 * self.heatmap_sigma))
        
        for (z, y, x) in coords:
            z, y, x = int(z), int(y), int(x)
            z_min, z_max = max(0, z - radius), min(Z, z + radius + 1)
            y_min, y_max = max(0, y - radius), min(Y, y + radius + 1)
            x_min, x_max = max(0, x - radius), min(X, x + radius + 1)
            
            if z_max <= z_min or y_max <= y_min or x_max <= x_min:
                continue
                
            zz, yy, xx = np.mgrid[z_min:z_max, y_min:y_max, x_min:x_max]
            dist_sq = ((zz - z)**2) + ((yy - y)**2) + ((xx - x)**2)
            blob = np.exp(-dist_sq / (2 * self.heatmap_sigma**2))
            
            heatmap[z_min:z_max, y_min:y_max, x_min:x_max] = np.maximum(
                heatmap[z_min:z_max, y_min:y_max, x_min:x_max], blob
            )
        return heatmap

    def __getitem__(self, idx):
        ds, t = self.valid_frames[idx]
        zarr_path = os.path.join(self.data_dir, f"{ds}.zarr")
        
        # Read metadata
        with open(os.path.join(zarr_path, '0', 'zarr.json')) as f:
            arr_meta = json.load(f)
        shape = tuple(arr_meta['shape'])
        dtype = np.dtype(arr_meta['data_type'])
        vol_shape = shape[1:]
        
        # Read specific frame
        chunk_path = os.path.join(zarr_path, '0', 'c', str(t), '0', '0', '0')
        with open(chunk_path, 'rb') as fh:
            compressed = fh.read()
        decompressed = blosc2.decompress(compressed)
        vol = np.frombuffer(decompressed, dtype=dtype).reshape(vol_shape).astype(np.float32)
        
        # Normalize volume
        vol_min, vol_max = vol.min(), vol.max()
        if vol_max > vol_min:
            vol = (vol - vol_min) / (vol_max - vol_min)
            
        # Get coordinates for this frame
        ds_data = self.df[(self.df['dataset'] == ds) & (self.df['t'] == t)]
        coords = ds_data[['z', 'y', 'x']].values
        
        # Generate target heatmap
        target = self._generate_heatmap(coords, vol_shape)
        
        # Random Crop to Patch Size
        Z, Y, X = vol_shape
        pz, py, px = self.patch_size
        
        # Ensure patch size is smaller than volume
        pz = min(pz, Z)
        py = min(py, Y)
        px = min(px, X)
        
        # Bias the crop to contain a cell 80% of the time
        if len(coords) > 0 and np.random.rand() < 0.8:
            cz, cy, cx = coords[np.random.randint(0, len(coords))]
            start_z = int(np.clip(cz - pz//2, 0, Z - pz))
            start_y = int(np.clip(cy - py//2, 0, Y - py))
            start_x = int(np.clip(cx - px//2, 0, X - px))
        else:
            start_z = np.random.randint(0, Z - pz + 1)
            start_y = np.random.randint(0, Y - py + 1)
            start_x = np.random.randint(0, X - px + 1)
            
        vol_patch = vol[start_z:start_z+pz, start_y:start_y+py, start_x:start_x+px]
        target_patch = target[start_z:start_z+pz, start_y:start_y+py, start_x:start_x+px]
        
        # Add channel dimension
        vol_patch = np.expand_dims(vol_patch, axis=0)
        target_patch = np.expand_dims(target_patch, axis=0)
        
        return torch.tensor(vol_patch), torch.tensor(target_patch)



In [ ]:
# Create DataLoader
train_ds = CellTrackingDataset(TRAIN_DIR, PATCH_SIZE, HEATMAP_SIGMA)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)



In [ ]:
# Model Definition
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=1,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
    norm="batch",
).to(device)

loss_function = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)



In [ ]:
# Training Loop
best_loss = float('inf')

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    step = 0
    
    # We take a small subset of frames per epoch to speed up iteration
    # In a full run, you'd iterate over the whole dataset
    for batch_data in train_loader:
        step += 1
        inputs, targets = batch_data[0].to(device), batch_data[1].to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_function(outputs, targets)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
        if step % 50 == 0:
            print(f"Epoch {epoch+1}/{EPOCHS}, Step {step}/{len(train_loader)}, Loss: {loss.item():.4f}")
            
    epoch_loss /= step
    print(f"--- Epoch {epoch+1} Average Loss: {epoch_loss:.4f} ---")
    
    if epoch_loss < best_loss:
        best_loss = epoch_loss
        torch.save(model.state_dict(), "model_best.pth")
        print("Saved new best model!")

print("Training complete.")

